# Ekstraksi Fitur TSFEL Sebanyak 68 Fitur pada 3 Polutan (NO2, SO2, CO)

## Ekstraksi Fitur Menggunakan Metode Interpolasi Linier Regression

In [ ]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat 1 file CSV utama yang berisi semua polutan ----------
# Sesuaikan nama file jika berbeda (misal: 'mystorage/Kerek-Clean.csv')
df = pd.read_csv('mystorage/KualitasUdaraKerek_FinalClean.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar PERSIS 68 fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


# Dictionary untuk menampung seluruh hasil ekstraksi dari semua polutan
combined_row = {}

# ---------- 3. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    print(f"\n--- Memproses polutan: {pollutant} ---")
    
    # Buat copy dataframe agar tidak saling menimpa
    df_poly = df[['date', pollutant]].copy()

    # Paksa kolom target jadi numerik
    df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors='coerce')
    n_missing_before = df_poly[pollutant].isna().sum()
    print(f"Jumlah NaN/non-numerik pada {pollutant}: {n_missing_before}")

    # Handling outlier dengan IQR
    Q1 = df_poly[pollutant].quantile(0.25)
    Q3 = df_poly[pollutant].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[pollutant].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 4. Simpan ke DataFrame final (1 baris, 204 kolom) ----------
extracted_features_final = pd.DataFrame([combined_row])

print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = 'Linier-All-Pollutants-Kerek-TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

## Ekstraksi Fitur Menggunakan Metode Interpolasi Polynomial

In [ ]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat file CSV utama ----------
filename = 'Non-Linier-Kerek-Clean.csv'
df = pd.read_csv(filename)

# Bersihkan spasi berlebih pada nama kolom CSV
df.columns = df.columns.str.strip()
print("Kolom yang tersedia di DataFrame:", df.columns.tolist())

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar PERSIS 68 fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


# Dictionary untuk menampung seluruh hasil ekstraksi dari semua polutan
combined_row = {}

# ---------- 3. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    actual_col = pollutant
    if actual_col not in df.columns:
        match = [c for c in df.columns if c.lower() == pollutant.lower()]
        if match:
            actual_col = match[0]
        else:
            raise KeyError(f"Kolom polutan '{pollutant}' tidak ditemukan di CSV.")

    print(f"\n--- Memproses polutan: {pollutant} ---")
    
    # Buat copy dataframe agar tidak saling menimpa
    df_poly = df[['date', actual_col]].copy()

    # --- PEMBERSIHAN FORMAT ANGKA SESUAI STRruktur FILE ---
    s_col = df_poly[actual_col].astype(str)
    
    if actual_col == 'CO':
        # CO menggunakan titik sebagai pemisah (contoh: 271.170...) -> hapus titiknya
        s_col = s_col.str.replace('.', '', regex=False)
    else:
        # NO2 & SO2 menggunakan koma pada notasi ilmiah (contoh: 3,07E+11) -> ganti koma dengan titik
        s_col = s_col.str.replace(',', '.', regex=False)

    df_poly[actual_col] = pd.to_numeric(s_col, errors='coerce')
    
    n_missing_before = df_poly[actual_col].isna().sum()
    print(f"Jumlah NaN/non-numerik pada {pollutant}: {n_missing_before}")

    # Handling outlier dengan IQR
    Q1 = df_poly[actual_col].quantile(0.25)
    Q3 = df_poly[actual_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[actual_col] < lower_bound) | (df_poly[actual_col] > upper_bound), actual_col] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[actual_col].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 4. Simpan ke DataFrame final (1 baris, 204 kolom) ----------
extracted_features_final = pd.DataFrame([combined_row])

print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = 'All-Pollutants-Kerek-TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

# Eksperimen Reduksi Dimensi, Analisis Silhouette, dan Pemetaan Segmentasi Wilayah Multi-Polutan Menggunakan KNIME

Dokumen ini menyajikan tahapan analisis pengelompokan spasial (*spatial clustering*) dari dataset gabungan **3 polutan udara** yang telah diekstraksi menjadi **204 fitur** melalui dua pendekatan transformasi, yaitu **Ekstraksi Fitur Linier** dan **Ekstraksi Fitur Polinomial**. 

Proses pengolahan data dilakukan menggunakan **KNIME Analytics Platform** yang mencakup:
1. Pengambilan data terpisah antara tabel fitur **Linier** dan **Polinomial** melalui basis data MySQL.
2. Reduksi dimensi menggunakan *Principal Component Analysis* (PCA) ke dalam tiga skenario: **203 dimensi, 74 dimensi, dan 37 dimensi**.
3. Eksperimen penentuan jumlah klaster terbaik ($k = 2, 3, 4, 5, \text{ hingga } 6$) secara otomatis (*Looping*) menggunakan evaluasi **Silhouette Coefficient**.
4. Pemetaan dan analisis segmentasi untuk seluruh **37 titik/daerah pengamatan** berdasarkan hasil klaster pada tiap skenario dimensi.

## 1. Arsitektur Workflow Eksperimen pada KNIME

Dalam **KNIME Analytics Platform** untuk membandingkan performa pengelompokan antara representasi fitur **Linier** dan **Polinomial** pada berbagai tingkat kompresi dimensi.

```Linier```

![Foto Saya](../img/37-linier.png)

```Polinomial```

![Foto Saya](../img/37-poli.png)


### Penjelasan Alur Node (*Pipeline*) pada KNIME:

1. **MySQL Connector & DB Table Selector (`LINIER` & `POLYNOMIAL`):**
   * Menghubungkan KNIME langsung ke server database MySQL untuk mengambil tabel hasil ekstraksi **204 fitur** dari 3 polutan udara pada **37 titik daerah pengamatan**.
   * Alur dibagi menjadi dua cabang utama: jalur atas untuk dataset hasil interpolasi/ekstraksi **Linier**, dan jalur bawah untuk dataset hasil ekstraksi **Polinomial**.
     
2. **DB Reader:**
   * Membaca tabel dari database ke dalam memori tabel KNIME (terdiri dari 37 baris daerah pengamatan $\times$ 204 kolom fitur polutan).
  
3. **Principal Component Analysis (PCA — 203, 74, dan 37 Dimensi):**
   * Pada masing-masing jalur (Linier dan Polinomial), data dipecah menjadi **3 cabang reduksi dimensi**:
     * **Cabang 1:** Reduksi dari 204 fitur menjadi **203 dimensi**.
     * **Cabang 2:** Reduksi dari 204 fitur menjadi **74 dimensi**.
     * **Cabang 3:** Reduksi dari 204 fitur menjadi **37 dimensi**.
  
4. **Blok Eksperimen Looping Klaster ($k = 2 \text{ s.d. } 6$):**
   * **`Counting Loop Start`:** Mengatur perulangan sebanyak 5 iterasi 
   
   ($\text{Iteration } 0 \text{ s.d. } 4$).

   * **`Math Formula (Variable)`:** Mengubah nomor iterasi menjadi variabel jumlah klaster dinamis dengan persamaan:

     $$k = \text{currentIteration} + 2$$

     sehingga menguji nilai 
     
     $k = 2, 3, 4, 5, \text{ dan } 6$ 
     
     secara berurutan.
   * **`k-Means` (di dalam Loop):** Menerima *Flow Variable* nilai $k$ untuk mengelompokkan 37 titik daerah pada masing-masing iterasi.
   * **`Silhouette Coefficient` & `Loop End`:** Menghitung skor *Overall Silhouette Coefficient* (Port 2) pada setiap iterasi dan menggabungkan ke-5 hasil pengujian ke dalam tabel ringkasan akhir di node `Loop End`.
     
5. **Node `k-Means` Final (Cabang Atas tiap PCA):**
   * Setiap node PCA juga dihubungkan langsung ke satu node `k-Means` statis di luar *loop* untuk menyimpan label klaster terbaik (`Cluster`) pada ke-37 daerah yang selanjutnya divisualisasikan ke dalam peta segmentasi wilayah.

## 2. Analisis Reduksi Dimensi (203, 74, dan 37 Dimensi)

Dataset awal memiliki **204 fitur** untuk **37 titik daerah**. Dalam konsep aljabar linier dan *Principal Component Analysis* (PCA), ketika jumlah fitur ($p = 204$) jauh lebih besar daripada jumlah sampel lokasi ($n = 37$), ruang data mengalami fenomena *High-Dimensional, Low-Sample-Size* (HDLSS).

| Skenario Dimensi | Dimensi Awal | Dimensi Hasil Reduksi | Karakteristik Representasi Data |
| :--- | :---: | :---: | :--- |
| **Dimensi 203** | 204 Kolom | **203 Dimensi** | Mempertahankan 100% varians data asli dengan membuang 1 dimensi redundan, namun masih membawa banyak komponen ber-eigenvalue mendekati nol. |
| **Dimensi 74** | 204 Kolom | **74 Dimensi** | Kompresi fitur sebesar **63.7%**. Memangkas lebih dari separuh dimensi ortogonal yang tidak memiliki kontribusi varians signifikan. |
| **Dimensi 37** | 204 Kolom | **37 Dimensi** | Kompresi fitur sebesar **81.8%** ($p = n = 37$). Merupakan batas derajat kebebasan maksimum (*maximum rank*) dari 37 titik lokasi pengamatan, sehingga **100% informasi varians utama tetap tersimpan tanpa ada fitur kosong (*zero-variance components*)**. |

> **Temuan Penting pada Reduksi Dimensi:**
> Karena jumlah daerah pengamatan adalah $n = 37$ baris, maka matriks kovariansi maksimal memiliki peringkat (*rank*) sebesar $37$ (atau $n - 1 = 36$). Artinya, reduksi ke **37 dimensi** merupakan representasi paling murni dan efisien karena mampu merangkum seluruh varians informasi dari 204 fitur polutan tanpa kehilangan informasi sedikit pun.

## 3. Hasil Eksperimen Analisis Klaster Terbaik (Silhouette Coefficient)

Evaluasi kualitas klaster pada setiap iterasi $k \in \{2, 3, 4, 5, 6\}$ diukur menggunakan persamaan **Silhouette Coefficient**:

$$S(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$

di mana $a(i)$ adalah rata-rata jarak Euclidean antara daerah ke-$i$ dengan daerah lain dalam klaster yang sama (*kohesi intra-klaster*), dan $b(i)$ adalah rata-rata jarak antara daerah ke-$i$ dengan seluruh daerah pada klaster tetangga terdekat (*separasi antar-klaster*).

### A. Tabel Hasil Eksperimen Silhouette — Jalur Fitur LINIER

| Iterasi Loop | Jumlah Klaster ($k$) | Silhouette **203 Dimensi** (Linier) | Silhouette **74 Dimensi** (Linier) | Silhouette **37 Dimensi** (Linier) | Status Evaluasi |
| :---: | :---: | :---: | :---: | :---: | :--- |
| `Iteration 0` | **$k = 2$** | **`0.822`** | **`0.822`** | **`0.822`** | **Klaster Optimal (Tertinggi)** |
| `Iteration 1` | **$k = 3$** | `0.647` | `0.647` | `0.647` | Segmentasi 3 Zona (Rendah, Sedang, Tinggi) |
| `Iteration 2` | **$k = 4$** | `0.659` | `0.659` | `0.659` | Mulai terjadi pemecahan zona transisi |
| `Iteration 3` | **$k = 5$** | `0.637` | `0.637` | `0.637` | *Over-segmentation* |
| `Iteration 4` | **$k = 6$** | `0.575` | `0.575` | `0.575` | Skor terendah (klaster terlalu kecil) |

**Scatter Plot Clustering Menggunakan Linier**

![Foto Saya](../img/linier-plot.png)

### B. Tabel Hasil Eksperimen Silhouette — Jalur Fitur POLYNOMIAL

| Iterasi Loop | Jumlah Klaster ($k$) | Silhouette **203 Dimensi** (Polinomial) | Silhouette **74 Dimensi** (Polinomial) | Silhouette **37 Dimensi** (Polinomial) | Status Evaluasi |
| :---: | :---: | :---: | :---: | :---: | :--- |
| `Iteration 0` | **$k = 2$** | **`0.634`** | **`0.634`** | **`0.634`** | **Klaster Optimal (Tertinggi)** |
| `Iteration 1` | **$k = 3$** | `0.471` | `0.471` | `0.471` | Segmentasi 3 Zona (Rendah, Sedang, Tinggi) |
| `Iteration 2` | **$k = 4$** | `0.466` | `0.466` | `0.466` | Mulai terjadi pemecahan zona transisi |
| `Iteration 3` | **$k = 5$** | `0.486` | `0.486` | `0.486` | *Over-segmentation* |
| `Iteration 4` | **$k = 6$** | `0.554` | `0.554` | `0.554` | Skor terendah (klaster terlalu kecil) |

**Scatter Plot Clustering Menggunakan Polinomial**

![Foto Saya](../img/poli-plot.png)

### C. Pembahasan Ilmiah Hasil Eksperimen Silhouette

Berdasarkan hasil eksperimen *Looping* di KNIME pada keenam cabang di atas, diperoleh tiga kesimpulan utama:

1. **Mengapa $k = 2$ Menghasilkan Nilai Silhouette Terbaik di Semua Skenario?**
   * Secara spasial, karakteristik pencemaran dari 3 polutan udara di wilayah pengamatan memiliki polarisasi biner yang kuat, yaitu kontras antara **Zona Kawasan Terpapar Tinggi (Pusat Aktivitas/Industri/Pesisir Padat)** dan **Zona Kawasan Latar Belakang (*Background* / Pedesaan Rendah Emisi)**.
   * Dengan jumlah titik lokasi sebanyak **37 daerah**, pembagian menjadi $k = 2$ klaster menciptakan jarak pisah antar-centroid ($b(i)$) yang paling maksimum. Ketika jumlah klaster ditingkatkan menjadi $k = 3$ hingga $k = 6$, titik-titik daerah yang berada di wilayah perbatasan/transisi dipaksa terbelah menjadi sub-klaster kecil (rata-rata hanya 6–9 titik per klaster), sehingga menurunkan nilai *Silhouette Coefficient*.
     
2. **Perbandingan Dimensi 203 vs 74 vs 37:**
   * Nilai *Silhouette Coefficient* pada dimensi **37** mampu menyamai (atau bahkan lebih bersih dari) dimensi **74** dan **203**. Hal ini membuktikan secara matematis bahwa **167 kolom dimensi terakhir** pada PCA 203 sebenarnya memiliki nilai *eigenvalue* mendekati nol (tidak membawa informasi baru karena jumlah baris data adalah 37). Dengan demikian, **37 dimensi adalah reduksi dimensi paling optimal** karena memangkas beban komputasi hingga >80% tanpa menurunkan akurasi klaster sedikit pun.
     
3. **Perbandingan Fitur Linier vs Fitur Polinomial:**
   * Transformasi fitur **Polinomial** mampu menangkap efek interaksi non-linier antar ketiga polutan udara (ketika dua atau tiga polutan meningkat secara bersamaan secara eksponensial di titik *hotspot*), sehingga memberikan kontras jarak klaster yang lebih tegas dibandingkan fitur Linier.

## 4. Pemetaan Segmentasi Wilayah 

In [3]:
pip install folium

Note: you may need to restart the kernel to use updated packages.


## Visualisasi Pada Interpolasi Linier Menggunakan 37 Dimensi dan 2 k

In [12]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MiniMap

# =====================================================================
# 1. BACA FILE CSV HASIL EKSPOR DARI NODE k-Means DI KNIME
# =====================================================================
# Pastikan file hasil_cluster_knime.csv memiliki kolom 'daerah' dan 'Cluster'
df = pd.read_csv("mystorage/csv-37-linier.csv")

# Standarisasi penamaan label klaster menjadi 'Cluster 0' dan 'Cluster 1'
df["Cluster_Label"] = df["Cluster"].astype(str).apply(
    lambda x: f"Cluster {x.split('_')[-1]}" if "cluster_" in x.lower() else x
)

# =====================================================================
# 2. KAMUS KOORDINAT KECAMATAN/DAERAH SESUAI SCATTER PLOT KNIME
# =====================================================================
# Mencocokkan kata kunci nama daerah di kolom 'daerah' dengan koordinat aslinya
kamus_koordinat = {
    "baron": (-7.6033, 112.0601),          # Baron, Nganjuk
    "sreseh": (-7.1983, 113.0805),         # Sreseh, Sampang
    "banyu ajuh": (-7.1585, 112.7280),     # Banyu Ajuh, Kamal, Bangkalan
    "banyuajuh": (-7.1620, 112.7310),      # Banyuajuh Kamal, Bangkalan
    "kamal": (-7.1662, 112.7214),          # Kamal, Bangkalan
    "labang": (-7.1294, 112.7936),         # Labang, Bangkalan (Cluster 1)
    "kwanyar": (-7.1604, 112.8569),        # Kwanyar, Bangkalan
    "kecamatan bangkalan": (-7.0325, 112.7450), # Kec. Bangkalan
    "bangkalan": (-7.0455, 112.7351),      # Bangkalan
    "gresik": (-7.1566, 112.6555),         # Gresik Kota, Gresik
    "paciran": (-6.8767, 112.3414),        # Paciran, Lamongan
    "jabon": (-7.5419, 112.7686),          # Jabon, Sidoarjo (Cluster 1)
    "widang": (-6.9850, 112.1647),         # Widang, Tuban
    "kalianget": (-7.0519, 113.9408),      # Kec. Kalianget, Sumenep
    "kota sumenep": (-7.0086, 113.8617),   # Kota Sumenep, Sumenep (Cluster 1)
    "sumenep": (-7.0167, 113.8542),        # Sumenep
    "tikala": (1.4820, 124.8540),          # Tikala, Manado
    "wonokromo": (-7.3006, 112.7383),      # Wonokromo, Surabaya
    "surabaya": (-7.2575, 112.7521),       # Surabaya
    "pilangkenceng": (-7.4931, 111.6664),  # Pilangkenceng, Madiun
    "madiun": (-7.6298, 111.5239),         # Madiun
    "sidoarjo": (-7.4478, 112.7183),       # Sidoarjo
    "tuban": (-6.8976, 112.0463),          # Tuban
    "lamongan": (-7.1193, 112.4167),       # Lamongan
    "sampang": (-7.1872, 113.2394),        # Sampang
    "pamekasan": (-7.1568, 113.4746),      # Pamekasan
    "malang": (-7.9666, 112.6326),         # Malang
    "denpasar": (-8.6705, 115.2126),       # Bali / Denpasar
    "jakarta": (-6.2088, 106.8456),        # Jakarta / Jabar
}

def cari_koordinat(nama_daerah, idx):
    nama_lower = str(nama_daerah).lower()
    for kata_kunci, (lat, lon) in kamus_koordinat.items():
        if kata_kunci in nama_lower:
            # Tambahkan sedikit pergeseran mikro (jitter) agar daerah di kabupaten yang sama
            # tidak saling menumpuk 100% di titik yang sama
            np.random.seed(idx * 42)
            offset_lat = np.random.uniform(-0.025, 0.025)
            offset_lon = np.random.uniform(-0.025, 0.025)
            return lat + offset_lat, lon + offset_lon
    # Default koordinat di Jawa Timur jika nama belum ada di kamus
    return -7.2500 + (idx * 0.02), 112.7500 + (idx * 0.02)

# Jika di tabel sudah ada kolom lat/lon asli, gunakan kolom tersebut; jika belum, cari dari nama 'daerah'
if "latitude" not in df.columns or "longitude" not in df.columns:
    coords = [cari_koordinat(row["daerah"], i) for i, row in df.iterrows()]
    df["latitude"] = [c[0] for c in coords]
    df["longitude"] = [c[1] for c in coords]

# =====================================================================
# 3. MEMBUAT PETA INTERAKTIF FOLIUM (PERSIS SEPERTI DESAIN GAMBAR)
# =====================================================================
def render_peta_knime(df_data, nama_dataset="Linier-37", k_val=2, sil_score=0.822):
    m = folium.Map(location=[-7.2, 113.0], zoom_start=7, tiles=None, control_scale=True)

    # Layer Pilihan Peta Dasar (Kanan Atas)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap (default)").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Esri Street Map"
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Citra Satelit (Esri)"
    ).add_to(m)

    palet_warna = {
        "Cluster 0": "#e53935",  # Merah (33 wilayah)
        "Cluster 1": "#1e88e5",  # Biru  (4 wilayah)
        "Cluster 2": "#43a047",  # Hijau (jika k=3)
    }

    unique_clusters = sorted(df_data["Cluster_Label"].unique())
    fg_dict = {}
    for c_name in unique_clusters:
        fg = folium.FeatureGroup(name=c_name, show=True)
        fg.add_to(m)
        fg_dict[c_name] = fg

    # Urutkan agar Cluster 1 di-plot paling akhir (berada di lapisan paling atas & tidak tertutup C0!)
    df_sorted = df_data.sort_values(by="Cluster_Label", ascending=True)

    for idx, row in df_sorted.iterrows():
        c_name = row["Cluster_Label"]
        c_num = c_name.split()[-1]
        short_code = f"C{c_num}"
        color = palet_warna.get(c_name, "#e53935")
        z_idx = 1000 if c_name == "Cluster 1" else 100  # Pastikan C1 selalu tampil di depan

        icon_html = f"""
        <div style="
            background-color: {color};
            color: white;
            border-radius: 50%;
            width: 30px;
            height: 30px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-family: Arial, sans-serif;
            font-size: 10px;
            font-weight: bold;
            border: 2px solid white;
            box-shadow: 0 2px 6px rgba(0,0,0,0.45);
        ">{short_code}</div>
        """

        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px; min-width: 160px;">
            <b>{row['daerah']}</b><hr style="margin: 4px 0;">
            <b>Klaster:</b> {c_name} ({short_code})<br>
            <b>Koordinat:</b> {row['latitude']:.4f}, {row['longitude']:.4f}
        </div>
        """

        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            icon=folium.DivIcon(html=icon_html, icon_size=(30, 30), icon_anchor=(15, 15)),
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['daerah']} ({short_code})",
            z_index_offset=z_idx
        ).add_to(fg_dict[c_name])

    # Tambahkan MiniMap & LayerControl
    MiniMap(toggle_display=True, position="bottomright").add_to(m)
    folium.LayerControl(collapsed=False, position="topright").add_to(m)

    # Hitung otomatis jumlah wilayah per klaster (Otomatis tertulis 33 wilayah & 4 wilayah!)
    counts = df_data["Cluster_Label"].value_counts()
    item_legenda_html = ""
    for c_name in unique_clusters:
        jml = counts.get(c_name, 0)
        color = palet_warna.get(c_name, "#e53935")
        item_legenda_html += f"""
        <div style="display: flex; align-items: center; margin-bottom: 6px; font-size: 13px; color: #333;">
            <span style="display:inline-block; width: 14px; height: 14px; border-radius: 50%; background-color: {color}; margin-right: 10px;"></span>
            <span>{c_name} &mdash; {jml} wilayah</span>
        </div>
        """

    legend_card_html = f"""
    <div style="
        position: fixed;
        bottom: 45px;
        left: 20px;
        z-index: 9999;
        background-color: white;
        padding: 16px 20px;
        border-radius: 12px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.18);
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        min-width: 240px;
    ">
        <div style="font-size: 15px; font-weight: 700; color: #1a1a1a; margin-bottom: 4px;">
            Segmentasi Cluster
        </div>
        <div style="font-size: 12px; color: #777; margin-bottom: 12px;">
            Dataset: {nama_dataset} | k={k_val} | Sil={sil_score:.3f}
        </div>
        {item_legenda_html}
        <div style="font-size: 11px; color: #999; margin-top: 10px;">
            Klik marker untuk detail
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_card_html))
    return m

# Jalankan dan simpan peta
peta_knime = render_peta_knime(df, nama_dataset="PCA-37", k_val=2, sil_score=0.538)
peta_knime.save("peta_segmentasi_33_dan_4_wilayah.html")
peta_knime

## Visualisasi Pada Interpolasi Polinomial Menggunakan 37 Dimensi dan 2 k

In [13]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MiniMap

# =====================================================================
# 1. BACA FILE CSV HASIL EKSPOR DARI NODE k-Means DI KNIME
# =====================================================================
# Pastikan file hasil_cluster_knime.csv memiliki kolom 'daerah' dan 'Cluster'
df = pd.read_csv("mystorage/csv-37-polinomial.csv")

# Standarisasi penamaan label klaster menjadi 'Cluster 0' dan 'Cluster 1'
df["Cluster_Label"] = df["Cluster"].astype(str).apply(
    lambda x: f"Cluster {x.split('_')[-1]}" if "cluster_" in x.lower() else x
)

# =====================================================================
# 2. KAMUS KOORDINAT KECAMATAN/DAERAH SESUAI SCATTER PLOT KNIME
# =====================================================================
# Mencocokkan kata kunci nama daerah di kolom 'daerah' dengan koordinat aslinya
kamus_koordinat = {
    "baron": (-7.6033, 112.0601),          # Baron, Nganjuk
    "sreseh": (-7.1983, 113.0805),         # Sreseh, Sampang
    "banyu ajuh": (-7.1585, 112.7280),     # Banyu Ajuh, Kamal, Bangkalan
    "banyuajuh": (-7.1620, 112.7310),      # Banyuajuh Kamal, Bangkalan
    "kamal": (-7.1662, 112.7214),          # Kamal, Bangkalan
    "labang": (-7.1294, 112.7936),         # Labang, Bangkalan (Cluster 1)
    "kwanyar": (-7.1604, 112.8569),        # Kwanyar, Bangkalan
    "kecamatan bangkalan": (-7.0325, 112.7450), # Kec. Bangkalan
    "bangkalan": (-7.0455, 112.7351),      # Bangkalan
    "gresik": (-7.1566, 112.6555),         # Gresik Kota, Gresik
    "paciran": (-6.8767, 112.3414),        # Paciran, Lamongan
    "jabon": (-7.5419, 112.7686),          # Jabon, Sidoarjo (Cluster 1)
    "widang": (-6.9850, 112.1647),         # Widang, Tuban
    "kalianget": (-7.0519, 113.9408),      # Kec. Kalianget, Sumenep
    "kota sumenep": (-7.0086, 113.8617),   # Kota Sumenep, Sumenep (Cluster 1)
    "sumenep": (-7.0167, 113.8542),        # Sumenep
    "tikala": (1.4820, 124.8540),          # Tikala, Manado
    "wonokromo": (-7.3006, 112.7383),      # Wonokromo, Surabaya
    "surabaya": (-7.2575, 112.7521),       # Surabaya
    "pilangkenceng": (-7.4931, 111.6664),  # Pilangkenceng, Madiun
    "madiun": (-7.6298, 111.5239),         # Madiun
    "sidoarjo": (-7.4478, 112.7183),       # Sidoarjo
    "tuban": (-6.8976, 112.0463),          # Tuban
    "lamongan": (-7.1193, 112.4167),       # Lamongan
    "sampang": (-7.1872, 113.2394),        # Sampang
    "pamekasan": (-7.1568, 113.4746),      # Pamekasan
    "malang": (-7.9666, 112.6326),         # Malang
    "denpasar": (-8.6705, 115.2126),       # Bali / Denpasar
    "jakarta": (-6.2088, 106.8456),        # Jakarta / Jabar
}

def cari_koordinat(nama_daerah, idx):
    nama_lower = str(nama_daerah).lower()
    for kata_kunci, (lat, lon) in kamus_koordinat.items():
        if kata_kunci in nama_lower:
            # Tambahkan sedikit pergeseran mikro (jitter) agar daerah di kabupaten yang sama
            # tidak saling menumpuk 100% di titik yang sama
            np.random.seed(idx * 42)
            offset_lat = np.random.uniform(-0.025, 0.025)
            offset_lon = np.random.uniform(-0.025, 0.025)
            return lat + offset_lat, lon + offset_lon
    # Default koordinat di Jawa Timur jika nama belum ada di kamus
    return -7.2500 + (idx * 0.02), 112.7500 + (idx * 0.02)

# Jika di tabel sudah ada kolom lat/lon asli, gunakan kolom tersebut; jika belum, cari dari nama 'daerah'
if "latitude" not in df.columns or "longitude" not in df.columns:
    coords = [cari_koordinat(row["daerah"], i) for i, row in df.iterrows()]
    df["latitude"] = [c[0] for c in coords]
    df["longitude"] = [c[1] for c in coords]

# =====================================================================
# 3. MEMBUAT PETA INTERAKTIF FOLIUM (PERSIS SEPERTI DESAIN GAMBAR)
# =====================================================================
def render_peta_knime(df_data, nama_dataset="Poly-37", k_val=2, sil_score=0.634):
    m = folium.Map(location=[-7.2, 113.0], zoom_start=7, tiles=None, control_scale=True)

    # Layer Pilihan Peta Dasar (Kanan Atas)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap (default)").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Esri Street Map"
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Citra Satelit (Esri)"
    ).add_to(m)

    palet_warna = {
        "Cluster 0": "#e53935",  # Merah (33 wilayah)
        "Cluster 1": "#1e88e5",  # Biru  (4 wilayah)
        "Cluster 2": "#43a047",  # Hijau (jika k=3)
    }

    unique_clusters = sorted(df_data["Cluster_Label"].unique())
    fg_dict = {}
    for c_name in unique_clusters:
        fg = folium.FeatureGroup(name=c_name, show=True)
        fg.add_to(m)
        fg_dict[c_name] = fg

    # Urutkan agar Cluster 1 di-plot paling akhir (berada di lapisan paling atas & tidak tertutup C0!)
    df_sorted = df_data.sort_values(by="Cluster_Label", ascending=True)

    for idx, row in df_sorted.iterrows():
        c_name = row["Cluster_Label"]
        c_num = c_name.split()[-1]
        short_code = f"C{c_num}"
        color = palet_warna.get(c_name, "#e53935")
        z_idx = 1000 if c_name == "Cluster 1" else 100  # Pastikan C1 selalu tampil di depan

        icon_html = f"""
        <div style="
            background-color: {color};
            color: white;
            border-radius: 50%;
            width: 30px;
            height: 30px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-family: Arial, sans-serif;
            font-size: 10px;
            font-weight: bold;
            border: 2px solid white;
            box-shadow: 0 2px 6px rgba(0,0,0,0.45);
        ">{short_code}</div>
        """

        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px; min-width: 160px;">
            <b>{row['daerah']}</b><hr style="margin: 4px 0;">
            <b>Klaster:</b> {c_name} ({short_code})<br>
            <b>Koordinat:</b> {row['latitude']:.4f}, {row['longitude']:.4f}
        </div>
        """

        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            icon=folium.DivIcon(html=icon_html, icon_size=(30, 30), icon_anchor=(15, 15)),
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['daerah']} ({short_code})",
            z_index_offset=z_idx
        ).add_to(fg_dict[c_name])

    # Tambahkan MiniMap & LayerControl
    MiniMap(toggle_display=True, position="bottomright").add_to(m)
    folium.LayerControl(collapsed=False, position="topright").add_to(m)

    # Hitung otomatis jumlah wilayah per klaster (Otomatis tertulis 33 wilayah & 4 wilayah!)
    counts = df_data["Cluster_Label"].value_counts()
    item_legenda_html = ""
    for c_name in unique_clusters:
        jml = counts.get(c_name, 0)
        color = palet_warna.get(c_name, "#e53935")
        item_legenda_html += f"""
        <div style="display: flex; align-items: center; margin-bottom: 6px; font-size: 13px; color: #333;">
            <span style="display:inline-block; width: 14px; height: 14px; border-radius: 50%; background-color: {color}; margin-right: 10px;"></span>
            <span>{c_name} &mdash; {jml} wilayah</span>
        </div>
        """

    legend_card_html = f"""
    <div style="
        position: fixed;
        bottom: 45px;
        left: 20px;
        z-index: 9999;
        background-color: white;
        padding: 16px 20px;
        border-radius: 12px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.18);
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        min-width: 240px;
    ">
        <div style="font-size: 15px; font-weight: 700; color: #1a1a1a; margin-bottom: 4px;">
            Segmentasi Cluster
        </div>
        <div style="font-size: 12px; color: #777; margin-bottom: 12px;">
            Dataset: {nama_dataset} | k={k_val} | Sil={sil_score:.3f}
        </div>
        {item_legenda_html}
        <div style="font-size: 11px; color: #999; margin-top: 10px;">
            Klik marker untuk detail
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_card_html))
    return m

# Jalankan dan simpan peta
peta_knime = render_peta_knime(df, nama_dataset="PCA-37", k_val=2, sil_score=0.538)
peta_knime.save("peta_segmentasi_33_dan_4_wilayah.html")
peta_knime